In [1]:
import sys; sys.path.append('/Users/arnavshah/Code/dnaBLT/training/data/iterators')
from training.data.iterators.v_args import TrainArgs
from training.data.iterators.v_arrow_iterator import ArrowFileIterator
from training.data.iterators.v_preprocess_iterator import PreprocessIterator

train_args = TrainArgs()
# train_args.data.buffer_size = 64
dataloader = train_args.data.build_from_rank(0, 1, 0, 1, "train")

In [9]:
preprocess_iterator = dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter
batch = next(iter(preprocess_iterator))

In [13]:
batch = next(iter(dataloader))
batch

Batch(x=tensor([[71, 75, 71,  ..., 71, 75, 75],
        [75, 88, 71,  ..., 71, 88, 88],
        [69, 75, 71,  ..., 71, 75, 71],
        ...,
        [75, 71, 71,  ..., 75, 88, 69],
        [88, 88, 71,  ..., 88, 71, 69],
        [69, 69, 88,  ..., 71, 75, 75]]), y=tensor([[75, 71, 71,  ..., 75, 75,  2],
        [88, 71, 69,  ..., 88, 88, 75],
        [75, 71, 71,  ..., 75, 71, 71],
        ...,
        [71, 71, 88,  ..., 88, 69, 88],
        [88, 71, 69,  ..., 71, 69, 71],
        [69, 88, 71,  ..., 75, 75, 88]]), mask=tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]]), patch_lengths=tensor([[ 1,  1,  1,  ...,  0,  0,  0],
        [ 1,  1,  1,  ...,  0,  0,  0],
        [ 1,  1,  1,  ...,  0,  0,  0],
        ...,
 

In [ ]:
(batch.patch_lengths != 0).sum(dim=1)

# ~~0. Arrow iterator~~
# ~~1. Preprocess iterator~~
# ~~2. Sequence iterator~~
# ~~3. Packing iterator~~


# Transformer -> Attention + FFW

tensor([1609, 2123, 1210, 1008,  997, 2534, 2419, 2765, 2201, 2062, 1552, 1952,
        1805, 2465, 3469,  845, 1265, 1759, 2153, 1467, 1384, 1165, 1032, 1158,
         914, 1420, 2338, 1928, 1788, 2322, 3633, 3351, 1857, 3349, 4121, 4212,
        3061, 3674, 2971, 1593, 1549, 1632, 1568, 1315, 1840, 1879, 1478, 2086,
        4005, 1896, 1591, 2986, 2087, 2273, 3077, 2504, 1499, 1732, 1757, 1156,
        2026, 4108, 2335, 1386, 1285, 1668, 1612, 1285, 1630, 1297, 1480, 1452,
        1501, 1911, 2126, 1279, 1461,  808,  729,  798, 2250,  744,  719, 1345,
        1651, 3437, 2475, 1865, 4550, 3253, 3702, 1837, 1115, 1533, 1945, 2566,
        1605, 2150, 2830, 1021, 1073,  985, 1113, 1347, 2263, 1699, 2207, 1166,
        1054, 1279, 1493, 1387, 1379,  957, 1121, 1057, 1168, 1051,  801, 1372,
         865, 1256,  982,  913, 1027, 1233, 1178, 1186])

In [12]:
batch.patch_lengths[0, :1609]

tensor([1, 1, 1,  ..., 1, 1, 2])

In [ ]:
iterator = iter(dataloader)
for _ in range(500):
    next(iterator)

In [5]:
(500 * 16 * 4096) / sum(dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter.patch_lengths)

tensor(0.9949)

In [6]:
dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter.arrow_batch_iterator.current_batch_idx

98

In [ ]:
import torch
import numpy as np
from tqdm import trange

all_nonzero_patch_lengths = []
length_sum = 0
patch_sum = 0

for _ in trange(1000):
    big_batch = next(preprocess_iterator)
    patch_lengths = big_batch.patch_lengths
    length_sum += (patch_lengths != 0).sum()
    patch_sum += patch_lengths.sum()
    # Flatten, filter, convert to numpy, and append
    nonzero_patch_lengths = patch_lengths[patch_lengths != 0]
    all_nonzero_patch_lengths.append(nonzero_patch_lengths.cpu())  # ensure on CPU if tensor

print("Average patch size", patch_sum / length_sum)
# Concatenate all batches into a single tensor
all_nonzero_patch_lengths = torch.cat(all_nonzero_patch_lengths, dim=0)

# Now proceed with your logic
tensor_np = all_nonzero_patch_lengths.numpy()
max_patch = all_nonzero_patch_lengths.max().item()
value_range = np.arange(1, max_patch + 1)
counts = np.bincount(tensor_np, minlength=max_patch + 1)[1:]  # skip index 0
weighted = counts * value_range
cdf = weighted.cumsum() / weighted.sum()
idx_99 = (cdf > 0.99).argmax()

print(f"99% of the weighted patch length mass is covered by patch length: {value_range[idx_99]}")

100%|██████████| 1000/1000 [03:50<00:00,  4.34it/s]


Average patch size tensor(2.1598)
99% of the weighted patch length mass is covered by patch length: 250
